 to make an AI Customer Support assistant for an Airline Assitant Bot to get the ticket price and booking details 


Lets first divide it small parts:
1. Create the get_ticket_price to get the ticket price for each city
1. We need to add set_ticket_price function to update the ticket price
2. Creating booking function to add the necessary details-  the destination ,passenger_name, date
3. Creating booking function to fetch the booking id
4. Creating Json schemas to add the details in the structured manner for LLM to understand
5. Updating the handle_tool_calls function which will help the LLM to fetch the necessary function
6. Now making the multi-modal by adding image, audio in the Gradio interface


In [ ]:
import os 
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins with {openai_api_key[:8]}")

else:
    print("OpenAI API key not set")


MODEL = "gpt-4.1-mini"
openai=OpenAI()


In [ ]:
system_prompt= """
You are a helpful assitant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate, if you don't know the answer, say so.
"""

In [ ]:
# Using tools to add the price 

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price= ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"the price of a ticket to {destination_city} is {price}"

In [ ]:
get_ticket_price("Berlin")

In [ ]:
def set_ticket_price(destination_city, price):
    city= destination_city.lower()  # normalize city names
    ticket_prices[city]= price     # updating the ticket prices
    return f"Price for the {destination_city} has been set to {price}."
    

In [ ]:
set_ticket_price("London", "$900")

In [ ]:
get_ticket_price("London")

In [ ]:
#Step 2: adding booking details

from datetime import datetime
import uuid
import time

bookings = {}   # global database of bookings

def create_booking(destination_city, passenger_name, date):
    # 1. Validate date
    try:
        dt = datetime.fromisoformat(date)     # <-- dt is defined here
    except:
        return {"error": "Invalid date format. Use YYYY-MM-DD"}

    # 2 Validate price
    price = ticket_prices.get(destination_city.lower())   # <-- price defined here
    if price is None:
        return {"error": f"No price found for {destination_city}. Please set a price first."}

    # 3. Generate booking ID
    booking_id = str(uuid.uuid4())

    # 4. Create booking dictionary
    booking = {
        "id": booking_id,
        "destination_city": destination_city,
        "passenger_name": passenger_name,
        "date": dt.date().isoformat(),              # <-- dt used correctly
        "price": price,                              # <-- price used correctly
        "status": "CONFIRMED",
        "created_at": datetime.now().isoformat()     # human-readable timestamp
    }

    # 5. Save booking
    bookings[booking_id] = booking

    # 6. Create summary for the assistant
    summary = (
        f"Booking confirmed (ID: {booking_id}) for {passenger_name} "
        f"to {destination_city} on {booking['date']} at {price}."
    )

    return {"ok": True, "summary": summary, "booking": booking}


In [ ]:
create_booking("London", "Somya", "2025-12-25" )

In [ ]:
# Creating booking function to fetch the booking id
def get_booking(booking_id):
    booking = bookings.get(booking_id)
    if booking:
        return booking
    return {"error": f"No booking found for id {booking_id}"}


In [ ]:
get_booking('d1f7b4d4-b4e4-4772-9e72-791677a83d89')

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
# Adding the json schema so that LLM can read and understand the functions 

set_price_schema = {
    "name": "set_ticket_price",
    "description": "Set or update the ticket price for a city (admin action).",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "City to update (e.g. 'London')"
            },
            "price": {
                "type": "string",
                "description": "Price string including currency (e.g. '$799')"
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

create_booking_schema = {
    "name": "create_booking",
    "description": "Create a booking and return a confirmation summary and id.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "Destination city (e.g. 'Tokyo')"
            },
            "passenger_name": {
                "type": "string",
                "description": "Full passenger name"
            },
            "date": {
                "type": "string",
                "description": "Date in ISO format YYYY-MM-DD"
            },
        },
        "required": ["destination_city", "passenger_name", "date"],
        "additionalProperties": False
    }
}

get_booking_schema = {
    "name": "get_booking",
    "description": "Retrieve booking details by booking id.",
    "parameters": {
        "type": "object",
        "properties": {
            "booking_id": {
                "type": "string",
                "description": "Booking identifier returned by create_booking"
            }
        },
        "required": ["booking_id"],
        "additionalProperties": False
    }
}

# Put them all in one list and pass to the model:
functions = [
    price_function,
    set_price_schema,
    create_booking_schema,
    get_booking_schema
]


In [ ]:
# And this is included in a list of tools:

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_schema},
    {"type": "function", "function": create_booking_schema},
    {"type": "function", "function": get_booking_schema},
]


In [ ]:
tools

# Adding the image function called artist


In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response= openai.images.generate(
        model = "dall-e-3",
        prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
#image = artist("New York City")
#display(image)

In [ ]:
import os
import uuid

def talker(message):
    os.makedirs("audio", exist_ok=True)

    audio_path = f"audio/{uuid.uuid4()}.mp3"

    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message
    )

    # Save audio to file
    with open(audio_path, "wb") as f:
        f.write(response.content)

    return audio_path   # ✅ Gradio wants a file path



In [ ]:
talker("Hi there")

In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []

    for tool_call in message.tool_calls:
        func_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments or "{}")

        if func_name == "get_ticket_price":
            city = args["destination_city"]
            content = get_ticket_price(city)
            cities.append(city)

        elif func_name == "set_ticket_price":
            city = args["destination_city"]
            content = set_ticket_price(city, args["price"])
            cities.append(city)

        elif func_name == "create_booking":
            result = create_booking(
                args["destination_city"],
                args["passenger_name"],
                args["date"]
            )
            content = result.get("summary", "Booking failed.")
            cities.append(args["destination_city"])

        elif func_name == "get_booking":
            booking = get_booking(args["booking_id"])
            content = json.dumps(booking) if booking else "No booking found."

        else:
            content = f"Unknown function {func_name}"

        responses.append({
            "role": "tool",
            "content": content,     # MUST be string
            "tool_call_id": tool_call.id
        })

    return responses, cities


In [ ]:
def chat(history):
    messages = [{"role": "system", "content": system_prompt}] + history

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        tool_responses, new_cities = handle_tool_calls_and_return_cities(assistant_message)
        messages.extend(tool_responses)
        cities.extend(new_cities)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    reply = response.choices[0].message.content
    history = history + [{"role": "assistant", "content": reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[-1])  # last city mentioned

    return history, voice, image



In [ ]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("sk", "bananas"))

In [ ]:
##gr.ChatInterface(fn=chat, type="messages").launch()

### NOW USING THE SAME FOR DATABASE

In [ ]:
import sqlite3

In [ ]:
DB ="prices.db"

with sqlite3.connect(DB) as conn:
    cursor= conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor= conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(), ))
        result =cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


In [ ]:
get_ticket_price("London")

In [ ]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1400.5, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()